# Numerical integration

This notebook explores Riemann sums, the trapezoid rule, and Simpson's rule through concrete integrals. We compare the approximations to the known answers and visualize how the error shrinks as the number of points increases.

### Overview of the methods

- **Riemann sums** partition the interval and add up rectangles. The **midpoint Riemann sum** usually gives better accuracy than the left/right variants.
- **Trapezoid rule** averages `f(x_i)` and `f(x_{i+1})`, so it exactly integrates linear functions.
- **Simpson's rule** fits parabolas to pairs of intervals and therefore integrates quadratics exactly and has higher-order convergence on smooth functions.

We will apply each method to familiar functions and compare both the estimated value and the convergence of the error as the grid is refined.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

assets_dir = root / "assets" / "figures"
assets_dir.mkdir(parents=True, exist_ok=True)

### Sin integral on [0, π]

The analytic integral of `sin(x)` over `[0, π]` is exactly 2. We compute the approximations using each method with a dense partition and visualize the area under `sin(x)` to connect the formula to the picture.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from calculus_with_python.grids import make_grid
from calculus_with_python.integration import integrate_f
from calculus_with_python.examples import sin

a, b = 0.0, float(np.pi)
x = make_grid(a, b, 400)
y = sin(x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y, label="sin(x)", color="C0")
ax.fill_between(x, 0.0, y, alpha=0.3, color="C0")
ax.set_title("Area under sin(x) on [0, π]")
ax.set_xlabel("x")
ax.set_ylabel("sin(x)")
ax.grid(True)
fig.savefig(assets_dir / "integration_area_sin.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig


In [ ]:
n = 801
true_value = 2.0

for method in ["riemann", "trapezoid", "simpson"]:
    approx = integrate_f(sin, a, b, n=n, method=method, riemann_side="midpoint")
    print(f"{method:9s}: {approx:.12f} (error {abs(approx - true_value):.2e})")


### Polynomial example

We also integrate a cubic polynomial on `[0, 1]`, where the true integral is `1/4`. Because Simpson's rule integrates quadratics exactly and extends that property, it should be particularly accurate for this case.

In [ ]:
from calculus_with_python.examples import poly

poly_fn = poly(0.0, 0.0, 0.0, 1.0)
poly_true = 0.25
x_poly = make_grid(0.0, 1.0, 300)
y_poly = poly_fn(x_poly)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_poly, y_poly, label="x^3", color="C2")
ax.fill_between(x_poly, 0.0, y_poly, alpha=0.25, color="C2")
ax.set_title("Area under x^3 on [0, 1]")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.grid(True)
fig.savefig(assets_dir / "integration_area_poly.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig


### Convergence study

We compute each method with progressively finer grids (`n` values range from 51 to 1601). The error should shrink at different rates: Riemann sums are `O(1/n)`, trapezoid is `O(1/n^2)` for smooth functions, and Simpson's rule is `O(1/n^4)` when the integrand is sufficiently smooth. Plotting the error vs. `n` on log scales makes the different slopes easy to see.

In [ ]:
import matplotlib.pyplot as plt
n_values = [51, 101, 201, 401, 801, 1601]
error_history = {"riemann": [], "trapezoid": [], "simpson": []}
true_sin = 2.0

for n in n_values:
    error_history["riemann"].append(abs(integrate_f(sin, a, b, n=n, method="riemann", riemann_side="midpoint") - true_sin))
    error_history["trapezoid"].append(abs(integrate_f(sin, a, b, n=n, method="trapezoid") - true_sin))
    error_history["simpson"].append(abs(integrate_f(sin, a, b, n=n, method="simpson") - true_sin))

fig, ax = plt.subplots(figsize=(7, 4))
for method, errors in error_history.items():
    ax.plot(n_values, errors, marker="o", label=method.capitalize())
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Error vs number of points for sin(x) integrals")
ax.set_xlabel("n (grid points)")
ax.set_ylabel("Absolute error")
ax.grid(True, which="both", linestyle=":")
ax.legend()
fig.savefig(assets_dir / "integration_error_vs_n.png", dpi=150, bbox_inches="tight")
plt.close(fig)
fig

### Symbolic definite integral

When SymPy is enabled, we can compare the symbolic definite integral with the numerical quadrature on a few familiar expressions and capture both the symbolic form and the error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from calculus_with_python.integration import integrate_f

try:
    from calculus_with_python.symbolic import lambdify_expr, symbolic_definite_integral, to_symbol
except ImportError:
    print('Install the optional symbolic extra (poetry install -E symbolic) to run this section.')
else:
    expressions = [
        ('sin(x)', 0.0, float(np.pi)),
        ('x**2', 0.0, 1.0),
        ('exp(-x**2)', -1.0, 1.0),
    ]

    fig_func, ax_func = plt.subplots(figsize=(7, 4))
    for expr, a, b in expressions:
        sym_fn = lambdify_expr(to_symbol(expr))
        x_plot = np.linspace(a, b, 400)
        ax_func.plot(x_plot, sym_fn(x_plot), label=expr)
    ax_func.set_title('Symbolic expressions over their domains')
    ax_func.set_xlabel('x')
    ax_func.set_ylabel('f(x)')
    ax_func.legend()
    ax_func.grid(True)
    fig_func.savefig(assets_dir / 'symbolic_integral_functions.png', dpi=150, bbox_inches='tight')
    plt.close(fig_func)

    ns = [101, 301, 601]
    errors = {expr: [] for expr, _, _ in expressions}
    for expr, a, b in expressions:
        sym_val = symbolic_definite_integral(expr, a, b)
        print(f'{expr}: symbolic integral = {sym_val}')
        numeric_fn = lambdify_expr(to_symbol(expr))
        for n in ns:
            numerical = integrate_f(numeric_fn, a, b, n=n, method='simpson')
            errors[expr].append(abs(float(sym_val) - numerical))
        print('	Simpson (n=601):', errors[expr][-1])

    fig_err, ax_err = plt.subplots(figsize=(7, 4))
    for expr in errors:
        ax_err.plot(ns, errors[expr], marker='o', label=expr)
    ax_err.set_xscale('log')
    ax_err.set_yscale('log')
    ax_err.set_title('Absolute error vs n (Simpson)')
    ax_err.set_xlabel('n')
    ax_err.set_ylabel('|symbolic - numerical|')
    ax_err.grid(True, which='both', linestyle=':')
    ax_err.legend()
    fig_err.savefig(assets_dir / 'symbolic_integral_error.png', dpi=150, bbox_inches='tight')
    plt.close(fig_err)
    fig_err

### Summary

Midpoint Riemann sums offer a straightforward `O(1/n)` convergence, trapezoid improves to `O(1/n^2)`, Simpson's rule reaches fourth-order convergence for smooth integrands, and the symbolic helpers admit exact definite integrals for comparison when available.